In [1]:
import openmc
import os
import numpy as np
import re
np.set_printoptions(precision=3)
#import openmc.data
os.system('rm materials.xml')
openmc.reset_auto_ids()

def shell_vol_calc(outer_radius,inner_radius):
    volume=4/3*np.pi*(outer_radius**3-inner_radius**3)
    return volume
def sphere_cap_vol(r,h):
    return 1/3*np.pi*h**2*(3*r-h)
def weight_to_atompercent(nuclides,weights):
    #Takes array of weights of different isotopes and returns a atom percent for the nuclide
    result = []
    nr_of_atoms = []
    
    for nuc,weight in zip(nuclides,weights):
        atomic_mass = openmc.data.atomic_mass(nuc)
        nr_of_atoms.append(weight/atomic_mass)

    total_nr_of_atoms = sum(nr_of_atoms)
    
    for atom in nr_of_atoms:
        result.append(atom/total_nr_of_atoms)
    return np.array(result)

def ao_to_wo(mat):
    wos = {}
    densities = mat.get_nuclide_atom_densities() #return atom densities in atom/b-cm (barn cm)
    for nuc_name,density in densities.items():
        mass = openmc.data.atomic_mass(nuc_name)
        wos[nuc_name]=mass*density/mat.density
    return wos
inner_fuel_r = 0
outer_fuel_r = 55.28815478052896
inner_moderator_height = 84

outer_moderator_r = 87
blanket_or = 100


blanket_volume = shell_vol_calc(blanket_or,outer_moderator_r)
moderator_volume = shell_vol_calc(inner_fuel_r,0)-sphere_cap_vol(inner_fuel_r,inner_moderator_height)\
    +shell_vol_calc(outer_moderator_r,outer_fuel_r)
fuel_volume = shell_vol_calc(outer_fuel_r,inner_fuel_r)

print(fuel_volume)

# Helper: make Li7-enriched lithium fluoride
LiF = openmc.Material(name='LiF')
LiF.add_elements_from_formula(formula='LiF',percent_type='ao',enrichment=99.9926,
                                 enrichment_target='Li7',enrichment_type='ao')

LiF_CA = openmc.Material(name='LiF_CA') #LiF with pure Li7
LiF_CA.add_elements_from_formula(formula='LiF',percent_type='ao',enrichment=99.999,
                                 enrichment_target='Li7',enrichment_type='ao')

BeF2 = openmc.Material(name='BeF2')
BeF2.add_elements_from_formula(formula='BeF2')

ZrF4 = openmc.Material(name='ZrF4')
ZrF4.add_elements_from_formula('ZrF4')  # 1 Zr : 4 F

ThF4 = openmc.Material(name='ThF4')
ThF4.add_elements_from_formula('ThF4')  # 1 Th : 4 F

UF4_LEU = openmc.Material(name='UF4_LEU')
UF4_LEU.add_elements_from_formula('UF4',percent_type='ao',enrichment=4.95
                                 ,enrichment_type='wo')
    #UF4_LEUeneriched to 4.95 % (LEU) and ao means by atom percent

UF4_A = openmc.Material(name='UF4_A')
#Using mass in kg from ORNL-TM-0611 p.7 to find atom percent 
atom_percent_U = weight_to_atompercent(['U234','U235','U236','U238'],[0.3,27,0.3,1.5])
print("ao for UF4 in fuel A",atom_percent_U)
atom_percent_UF4 = atom_percent_U*0.2
UF4_A.add_nuclide('U234',atom_percent_UF4[0])
UF4_A.add_nuclide('U235',atom_percent_UF4[1])
UF4_A.add_nuclide('U236',atom_percent_UF4[2])
UF4_A.add_nuclide('U238',atom_percent_UF4[3])
UF4_A.add_element('F',0.8)

UF4_B = openmc.Material(name='UF4_B')
#Using mass in kg from ORNL-TM-0611 p.7 to find atom percent 
atom_percent_U = weight_to_atompercent(['U234','U235','U236','U238'],[0.2,16.5,0.2,0.9])
print("ao for UF4 in fuel B",atom_percent_U)
atom_percent_UF4 = atom_percent_U*0.2
UF4_B.add_nuclide('U234',atom_percent_UF4[0])
UF4_B.add_nuclide('U235',atom_percent_UF4[1])
UF4_B.add_nuclide('U236',atom_percent_UF4[2])
UF4_B.add_nuclide('U238',atom_percent_UF4[3])
UF4_B.add_element('F',0.8)

UF4_C = openmc.Material(name='UF4_C')
#Using mass in kg from ORNL-TM-0611 p.7 to find atom percent 
atom_percent_U = weight_to_atompercent(['U234','U235','U236','U238'],[0.2,26.4,0.2,47.5])
print(f"ao for UF4 in fuel C {atom_percent_U}")
atom_percent_UF4 = atom_percent_U*0.2
UF4_C.add_nuclide('U234',atom_percent_UF4[0])
UF4_C.add_nuclide('U235',atom_percent_UF4[1])
UF4_C.add_nuclide('U236',atom_percent_UF4[2])
UF4_C.add_nuclide('U238',atom_percent_UF4[3])
UF4_C.add_element('F',0.8)



#Define Copenhagen Atomic (CA) fuel salt
fuel_CA_1 = openmc.Material.mix_materials([LiF_CA,UF4_LEU,ThF4],[0.73,0.23,0.04],'ao',name="Fuel_CA_1") #from CA_1
fuel_CA_1.volume = fuel_volume
fuel_CA_1.depletable = True


fuel_CA_2 = openmc.Material.mix_materials([LiF_CA,UF4_LEU],[0.73,0.27],'ao',name="Fuel_CA_2") #from CA_2 table 1
fuel_CA_2.set_density('g/cm3',4.905)
fuel_CA_2.volume = fuel_volume #245700 in cm^3 from CA-2 table 2
fuel_CA_2.temperature = 626.8 + 273.15 # in Kelvin
fuel_CA_2.depletable = True

#Define Copenhagen Atomic blanket salt
Blanket_CA = openmc.Material.mix_materials([LiF,ThF4],[0.70,0.30],'ao',name="Blanket_CA")
#from CA-2 table 1
Blanket_CA.set_density('g/cm3',4.626)
Blanket_CA.volume = blanket_volume # in cm^3 from CA-2
Blanket_CA.temperature = 626.8 + 273.15 # in Kelvin
Blanket_CA.depletable = True


#lb/ft3 to g/cm3
lbft_to_gcm = 1.6018463374/100
# Define Fuel A
print(sum([0.64969,0.292,0.05,0.00831]))
fuel_A = openmc.Material.mix_materials([LiF,BeF2,ZrF4,ThF4,UF4_A]
                                       ,[0.69987,0.237,0.05,0.01,0.00313],'ao',name='Fuel_A')
fuel_A.set_density('g/cm3',144.5*lbft_to_gcm)
fuel_A.volume = fuel_volume
fuel_A.depletable = True

# Fuel B
fuel_B = openmc.Material.mix_materials([LiF,BeF2,ZrF4,UF4_B],
                                       [0.66811,0.29,0.04,0.00189],'ao',name='Fuel_B')
fuel_B.set_density('g/cm3',134.5*lbft_to_gcm)
fuel_B.volume = fuel_volume
fuel_B.depletable = True

# Fuel C
fuel_C = openmc.Material.mix_materials([LiF,BeF2,ZrF4,UF4_C],
                                        [0.64969,0.292,0.05,0.00831],'ao',name='Fuel_C')
fuel_C.set_density('g/cm3', 142.7*lbft_to_gcm)
fuel_A.volume = fuel_volume
fuel_C.depletable = True

#Definition of air
N2 = openmc.Material(name='N2')
N2.add_elements_from_formula(formula='N2')

O2 = openmc.Material(name='O2')
O2.add_elements_from_formula(formula='O2')

Ar = openmc.Material(name='Ar')
Ar.add_elements_from_formula(formula='Ar')

CO2 = openmc.Material(name='CO2') #add_elements adds C0 so needed to be done "by hand"
#CO2.add_nuclide('C12', 1.0)
#CO2.add_nuclide('O16', 2.0)
CO2.add_elements_from_formula(formula='CO2') 

#With CO2
#air = openmc.Material.mix_materials([N2,O2,Ar,CO2],[0.7808,0.2095,0.0093,0.0004],'vo',name='Air')
#without CO2

air = openmc.Material.mix_materials([N2,O2,Ar],[0.7811,0.2096,0.0093],'vo',name='Air')



# Water 
water = openmc.Material(name="h2o")
water.add_nuclide('H1', 2.0)
water.add_nuclide('O16', 1.0)
water.set_density('g/cm3', 0.75)

water.add_s_alpha_beta('c_H_in_H2O')

# heavy_water 
heavy_water = openmc.Material(name="d2o")
heavy_water.add_nuclide('H2', 2.0)
heavy_water.add_nuclide('O16', 1.0)

heavy_water.set_density('g/cm3', 1.104) #from CA 2 table 1
heavy_water.volume = moderator_volume #2866100 in cm^3
heavy_water.temperature = 26.8 + 273.15 #in Kelvin
heavy_water.depletable = True

heavy_water.add_s_alpha_beta('c_D_in_D2O')


#UO2
uo2 = openmc.Material(name= "uo2",temperature=1200) #Available temperatures are 250, 294, 600, 900, 1200, 2500 K
# Add nuclides to uo2
uo2.add_nuclide('U235', 0.30)
uo2.add_nuclide('U238', 0.70)
uo2.add_nuclide('O16', 2.0)
uo2.set_density('g/cm3', 10.5)
uo2.volume = 1.0e4 
uo2.depletable = True


# Collect and export
materials = openmc.Materials([fuel_A, fuel_B, fuel_C, fuel_CA_1,fuel_CA_2, Blanket_CA, heavy_water, water,air, uo2])
materials.export_to_xml()
print(fuel_A)
print("✅ materials.xml has been created")

707921.1647499999
ao for UF4 in fuel A [0.01  0.928 0.01  0.051]
ao for UF4 in fuel B [0.011 0.928 0.011 0.05 ]
ao for UF4 in fuel C [0.003 0.358 0.003 0.636]
1.0
Material
	ID             =	13
	Name           =	Fuel_A
	Temperature    =	None
	Density        =	2.314667957543 [g/cm3]
	Volume         =	707921.1647499999 [cm^3]
	Depletable     =	True
	S(a,b) Tables  
	Nuclides       
	Li6            =	2.5895190000006466e-05 [ao]
	Li7            =	0.34990910481000004 [ao]
	F19            =	0.558439     [ao]
	Be9            =	0.07899999999999999 [ao]
	Zr90           =	0.005145     [ao]
	Zr91           =	0.0011220000000000004 [ao]
	Zr92           =	0.0017150000000000006 [ao]
	Zr94           =	0.0017380000000000008 [ao]
	Zr96           =	0.0002800000000000001 [ao]
	Th230          =	4.0000000000000025e-07 [ao]
	Th232          =	0.0019996000000000015 [ao]
	U234           =	6.485484884668586e-06 [ao]
	U235           =	0.0005812029061666951 [ao]
	U236           =	6.430406935507644e-06 [ao]
	U238   

In [2]:
inner_fuel_r = 0
outer_fuel_r = 50
V = 707921.16475
r = np.cbrt(3/(4*np.pi)*V)
outer_fuel_r = r
fuel_volume = shell_vol_calc(outer_fuel_r,inner_fuel_r)

print(r)
print(fuel_volume-707921.16475)

55.28815478052896
-1.1641532182693481e-10


In [3]:
wos = ao_to_wo(fuel_A)
symbol_to_element_name = {
    "H": "Hydrogen", "He": "Helium", "Li": "Lithium", "Be": "Beryllium",
    "B": "Boron", "C": "Carbon", "N": "Nitrogen", "O": "Oxygen",
    "F": "Fluorine", "Ne": "Neon", "Na": "Sodium", "Mg": "Magnesium",
    "Al": "Aluminum", "Si": "Silicon", "P": "Phosphorus", "S": "Sulfur",
    "Cl": "Chlorine", "Ar": "Argon", "K": "Potassium", "Ca": "Calcium",
    "Sc": "Scandium", "Ti": "Titanium", "V": "Vanadium", "Cr": "Chromium",
    "Mn": "Manganese", "Fe": "Iron", "Co": "Cobalt", "Ni": "Nickel",
    "Cu": "Copper", "Zn": "Zinc", "Ga": "Gallium", "Ge": "Germanium",
    "As": "Arsenic", "Se": "Selenium", "Br": "Bromine", "Kr": "Krypton",
    "Rb": "Rubidium", "Sr": "Strontium", "Y": "Yttrium", "Zr": "Zirconium",
    "Nb": "Niobium", "Mo": "Molybdenum", "Tc": "Technetium", "Ru": "Ruthenium",
    "Rh": "Rhodium", "Pd": "Palladium", "Ag": "Silver", "Cd": "Cadmium",
    "In": "Indium", "Sn": "Tin", "Sb": "Antimony", "Te": "Tellurium",
    "I": "Iodine", "Xe": "Xenon", "Cs": "Cesium", "Ba": "Barium",
    "La": "Lanthanum", "Ce": "Cerium", "Pr": "Praseodymium", "Nd": "Neodymium",
    "Pm": "Promethium", "Sm": "Samarium", "Eu": "Europium", "Gd": "Gadolinium",
    "Tb": "Terbium", "Dy": "Dysprosium", "Ho": "Holmium", "Er": "Erbium",
    "Tm": "Thulium", "Yb": "Ytterbium", "Lu": "Lutetium", "Hf": "Hafnium",
    "Ta": "Tantalum", "W": "Tungsten", "Re": "Rhenium", "Os": "Osmium",
    "Ir": "Iridium", "Pt": "Platinum", "Au": "Gold", "Hg": "Mercury",
    "Tl": "Thallium", "Pb": "Lead", "Bi": "Bismuth", "Po": "Polonium",
    "At": "Astatine", "Rn": "Radon", "Fr": "Francium", "Ra": "Radium",
    "Ac": "Actinium", "Th": "Thorium", "Pa": "Protactinium", "U": "Uranium",
    "Np": "Neptunium", "Pu": "Plutonium", "Am": "Americium", "Cm": "Curium",
    "Bk": "Berkelium", "Cf": "Californium", "Es": "Einsteinium", "Fm": "Fermium",
    "Md": "Mendelevium", "No": "Nobelium", "Lr": "Lawrencium", "Rf": "Rutherfordium",
    "Db": "Dubnium", "Sg": "Seaborgium", "Bh": "Bohrium", "Hs": "Hassium",
    "Mt": "Meitnerium", "Ds": "Darmstadtium", "Rg": "Roentgenium", "Cn": "Copernicium",
    "Nh": "Nihonium", "Fl": "Flerovium", "Mc": "Moscovium", "Lv": "Livermorium",
    "Ts": "Tennessine", "Og": "Oganesson"
}

def print_SR_in (mat,E_min=10,E_max=10000,state=0):
    #Method for making txt for .in files used by SR module
    #mat: The material used
    #E_min: minimum energy (keV)
    #E_max: maximumnergy (keV)
    #state: solid=0, gas=1
    empty_string = ''

    print('---Stopping/Range Input Data (Number-format: Period = Decimal Point)')
    print('---Output File Name')
    print(f'"{mat.name}"')
    print('---Ion(Z), Ion Mass(u)')
    print('2\t4')
    print('---Target Data: (Solid=0,Gas=1), Density(g/cm3), Compound Corr.')
    print(f'{state}\t {mat.density}\t 0')
    print('---Number of Target Elements')
    print(len(mat.nuclides))
    print('---Target Elements: (Z), Target name, Stoich, Target Mass(u)')
    for nuc in mat.nuclides:
        Stoich = nuc[1]
        Z, A, m = openmc.data.zam(nuc[0])
        symbol =re.match('([A-Za-z]+)',nuc[0])[0]
        print(f'{Z:<3}  "{symbol_to_element_name[symbol]}"  {empty_string:<8}\t {Stoich:<25}  {A}')
    print('---Output Stopping Units (1-8)')
    print('1') #Units eV / Angstrom
    print('---Ion Energy : E-Min(keV), E-Max(keV)')
    print(f'{E_min}\t{E_max}')
print_SR_in(fuel_A) 

---Stopping/Range Input Data (Number-format: Period = Decimal Point)
---Output File Name
"Fuel_A"
---Ion(Z), Ion Mass(u)
2	4
---Target Data: (Solid=0,Gas=1), Density(g/cm3), Compound Corr.
0	 2.314667957543	 0
---Number of Target Elements
15
---Target Elements: (Z), Target name, Stoich, Target Mass(u)
3    "Lithium"          	 2.5895190000006466e-05     6
3    "Lithium"          	 0.34990910481000004        7
9    "Fluorine"          	 0.558439                   19
4    "Beryllium"          	 0.07899999999999999        9
40   "Zirconium"          	 0.005145                   90
40   "Zirconium"          	 0.0011220000000000004      91
40   "Zirconium"          	 0.0017150000000000006      92
40   "Zirconium"          	 0.0017380000000000008      94
40   "Zirconium"          	 0.0002800000000000001      96
90   "Thorium"          	 4.0000000000000025e-07     230
90   "Thorium"          	 0.0019996000000000015      232
92   "Uranium"          	 6.485484884668586e-06      234
92   "Uraniu